# AgriMind AI: ML Engagement Prediction Model

This notebook contains the complete pipeline for training the RandomForestClassifier used in the AgriMind AI Microservice.

## Pipeline Overview
1. **Data Loading**: Ingest historical campaign data and grower profiles.
2. **Feature Engineering**: Extract temporal features (Month, Day of Week).
3. **Preprocessing**: Build a ColumnTransformer to handle missing values and encode categorical variables.
4. **Model Training**: Train a RandomForestClassifier to predict engagement.
5. **Serialization**: Export the trained pipeline as engagement_model.pkl using joblib for the FastAPI service.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

In [2]:
growers = pd.read_csv(
    "/kaggle/input/datasets/samasiayushman/sygetia/growers.csv"
)

campaigns = pd.read_csv(
    "/kaggle/input/datasets/samasiayushman/sygetia/whatsapp_campaign.csv"
)

print(growers.shape)
print(campaigns.shape)

(6000, 15)
(4479, 8)


In [3]:
df = campaigns.merge(
    growers,
    on="grower_id",
    how="left"
)

df.head()

,id,campaign_product,campaign_crop,grower_id,message_sent_date,delivered_status,opened_status,clicked_status,state,district,...,device_type,grower_age,gender,grower_crop_calendar,product_scan,product_name,product_scan_datetime,grower_farm_size,offline_campaign_attended,campaign_attendance_date
0,WAM_RABI25_00001,Tilt 250 EC,wheat,GRW_00001,2026-03-20,True,False,False,Rajasthan,Bharatpur,...,smartphone,67,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,3.54,False,NaN
1,WAM_RABI25_00002,Tilt 250 EC,wheat,GRW_00002,2026-03-31,True,False,False,Uttar Pradesh,Kanpur Nagar,...,smartphone,71,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,1.34,False,NaN
2,WAM_RABI25_00003,Tilt 250 EC,wheat,GRW_00003,2026-03-03,True,False,False,Punjab,Patiala,...,smartphone,52,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,0.55,True,2026-03-29
3,WAM_RABI25_00004,Tilt 250 EC,wheat,GRW_00004,2025-10-28,True,False,False,Rajasthan,Jaipur,...,smartphone,65,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,0.79,True,2026-01-31
4,WAM_RABI25_00005,Tilt 250 EC,wheat,GRW_00005,2025-12-08,True,False,False,Uttar Pradesh,Kanpur Nagar,...,smartphone,26,female,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,1.33,False,NaN


In [4]:
df["message_sent_date"] = pd.to_datetime(
    df["message_sent_date"]
)

df["month"] = df["message_sent_date"].dt.month

df["dayofweek"] = (
    df["message_sent_date"].dt.dayofweek
)

df["clicked_status"] = (
    df["clicked_status"]
    .astype(int)
)

df.head()

,id,campaign_product,campaign_crop,grower_id,message_sent_date,delivered_status,opened_status,clicked_status,state,district,...,gender,grower_crop_calendar,product_scan,product_name,product_scan_datetime,grower_farm_size,offline_campaign_attended,campaign_attendance_date,month,dayofweek
0,WAM_RABI25_00001,Tilt 250 EC,wheat,GRW_00001,2026-03-20,True,False,0,Rajasthan,Bharatpur,...,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,3.54,False,NaN,3,4
1,WAM_RABI25_00002,Tilt 250 EC,wheat,GRW_00002,2026-03-31,True,False,0,Uttar Pradesh,Kanpur Nagar,...,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,1.34,False,NaN,3,1
2,WAM_RABI25_00003,Tilt 250 EC,wheat,GRW_00003,2026-03-03,True,False,0,Punjab,Patiala,...,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,0.55,True,2026-03-29,3,1
3,WAM_RABI25_00004,Tilt 250 EC,wheat,GRW_00004,2025-10-28,True,False,0,Rajasthan,Jaipur,...,male,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,0.79,True,2026-01-31,10,1
4,WAM_RABI25_00005,Tilt 250 EC,wheat,GRW_00005,2025-12-08,True,False,0,Uttar Pradesh,Kanpur Nagar,...,female,"{""season"": ""Rabi_2025-26"", ""crop"": ""wheat"", ""s...",False,NaN,NaN,1.33,False,NaN,12,0


In [5]:
features = [
    "campaign_product",
    "campaign_crop",
    "state",
    "district",
    "language",
    "device_type",
    "grower_age",
    "gender",
    "grower_farm_size",
    "month",
    "dayofweek"
]

target = "clicked_status"

X = df[features]
y = df[target]

In [6]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
)

In [7]:
cat_features = [
    "campaign_product",
    "campaign_crop",
    "state",
    "district",
    "language",
    "device_type",
    "gender"
]

num_features = [
    "grower_age",
    "grower_farm_size",
    "month",
    "dayofweek"
]

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            cat_features
        ),
        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    )
                )
            ]),
            num_features
        )
    ]
)

In [9]:
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

In [10]:
pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", model)
])

pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['campaign_product',
                                                   'campaign_crop', 'state',
                                                   'district', 'language',
                                                   'device_type', 'gender']),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['grower_age',
                                                   'grower_farm_size', 'month',
                                                   'dayofweek'])])),
                ('model',
                 RandomForestClassifier(max_depth=8, n_estimators=50, n_jobs=-1,
                                        random_state=42))])

In [11]:
preds = pipeline.predict(
    X_test
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        preds
    )
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 0.9497767857142857
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       851
           1       0.00      0.00      0.00        45

    accuracy                           0.95       896
   macro avg       0.47      0.50      0.49       896
weighted avg       0.90      0.95      0.93       896



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
joblib.dump(
    pipeline,
    "engagement_model.pkl"
)

print("Saved!")

Saved!


In [13]:
sample = pd.DataFrame([
{
    "campaign_product": "Tilt 250 EC",
    "campaign_crop": "wheat",
    "state": "Rajasthan",
    "district": "Bharatpur",
    "language": "Hindi",
    "device_type": "smartphone",
    "grower_age": 67,
    "gender": "male",
    "grower_farm_size": 3.5,
    "month": 3,
    "dayofweek": 2
}
])

prob = pipeline.predict_proba(
    sample
)[0][1]

print(
    f"Predicted Engagement: {prob*100:.2f}%"
)

Predicted Engagement: 2.17%
